# 01_logging_sensitive — Logs as business events

"Payment declined by the limit" is a business event — plugins do not see it (they observe execution mechanics, not domain meaning). For such events an operation has `box` — a structured logger. Unlike `logger.info()`, a `box` event carries a **level** and is addressed to a **channel**, and who delivers it where is not decided by the operation code.

**What's new**

| Concept | Description |
|---------|-------------|
| `box.info / warning / critical(Channel, template, **kwargs)` | Level chosen by method name |
| `Channel.business` (combine with `\|`) | Semantic address of the event |
| `{%var.name}` / `{%var.name\|cyan}` | Template substitution (and color) |
| `{iif(...; red(...); green(...))}` | Conditional text with color branches |
| `PrivateAttr` + `@property` + `@sensitive` | A secret shown masked (e.g. `tok*****`) |

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio

from pydantic import Field, PrivateAttr

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.logging import Channel, ConsoleLogger, sensitive
from aoa.action_machine.logging.log_coordinator import LogCoordinator
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain, Params (with a masked secret), Result

`_api_token` is a `PrivateAttr` — it never reaches a template. The secret is exposed only through an `api_token` property whose getter is wrapped with `@sensitive` (so `@property` is outside, `@sensitive` inside), which masks it in logs.

In [ ]:
class RootDomain(BaseDomain):
    name = "root"
    description = "Root domain"


class LoginParams(BaseParams):
    username: str = Field(description="Username")
    amount: float = Field(description="Transaction amount")

    _api_token: str = PrivateAttr(default="")

    # Expose the secret only through a property; @sensitive masks it (e.g. tok*****).
    @property
    @sensitive(True, max_chars=3, char="*", max_percent=50)
    def api_token(self) -> str:
        return self._api_token

    def __init__(self, /, username: str, amount: float, api_token: str = "") -> None:
        super().__init__(username=username, amount=amount)
        self.__pydantic_private__["_api_token"] = api_token


class LoginResult(BaseResult):
    session_id: str = Field(description="Session ID")

## The action — substitution, `iif`, levels, masking

`box.info` with variable substitution and a conditional `iif`; `box.warning`/`box.critical` set the level by method name; `{%params.api_token}` prints the masked secret.

In [ ]:
@meta(description="Login", domain=RootDomain)
@check_roles(GuestRole)
class LoginAction(BaseAction[LoginParams, LoginResult]):

    @summary_aspect("Login")
    async def login_summary(self, params, state, box, connections):
        await box.info(
            Channel.business,
            "Login: user={%var.username}, amount={%var.amount}",
            username=params.username,
            amount=params.amount,
        )
        await box.info(
            Channel.business,
            "Info: {iif({%var.amount} > 1000; red('large transaction'); green('normal transaction'))}",
            amount=params.amount,
        )
        await box.warning(
            Channel.business,
            "Warning: amount {%var.amount|red} requires review",
            amount=params.amount,
        )
        await box.critical(
            Channel.business,
            "Critical: amount {%var.amount} requires review",
            amount=params.amount,
        )
        await box.info(Channel.business, "Token: {%params.api_token}")
        return LoginResult(session_id=f"sess-{params.username}-001")

## Run

Substitution (`user=alice`), the `iif` branch (500 ≤ 1000 → "normal transaction"), the `warning`/`critical` levels, and the masked token `tok*****` — three chars shown, the rest hidden, although `params` holds the full secret.

> In Colab, `await` works at top level — no `asyncio.run()`.

In [ ]:
async def main() -> None:
    machine = ActionProductMachine(log_coordinator=LogCoordinator(loggers=[ConsoleLogger()]))
    await machine.run(
        Context(),
        LoginAction(),
        params=LoginParams(username="alice", amount=500.0, api_token="tok-SUPER-SECRET-XYZ"),
    )


await main()